# Etude No. 1: Multi-Laminar Cortical AGSDR Scaffold

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/dev/tutorials/jaxfne_etude_no_1_multi_laminar_cortical_agsdr.ipynb)

**End-to-end jaxfne workflow:** Build a two-area laminar scaffold, simulate baseline/stimulus/tuned conditions, visualize spectrolaminar readouts, and export JSON-safe evidence.

## Scope Gates
- truth_mode: `truth_safe_unverified`
- claim_level: `computational_scaffold`  
- field_solver: NONE (proxy only)
- physical_amplitude_claims: DISABLED

## Installation

In [ ]:
!pip install -q jaxfne

## Imports & Setup

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import jaxfne as jtfne
import pandas as pd

print(f"jaxfne version: {jtfne.__version__}")

## Configuration

In [ ]:
RUN_ID = "etude_no_1"
SEED = 20260530
DURATION_MS = 300.0  # Fast smoke test
DT_MS = 0.1
N_PER_AREA = 40
OUTPUT_DIR = Path("outputs") / RUN_ID
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration: {N_PER_AREA} neurons/area, {DURATION_MS}ms duration")

## Step 1: Create Configuration & Model

In [ ]:
cfg = jtfne.default_spectrolaminar_config(
    areas=["V1", "V4"],
    n_per_area=N_PER_AREA,
    seed=SEED,
    duration_ms=DURATION_MS,
    dt_ms=DT_MS,
)
print(f"✓ Configuration created")

model = jtfne.construct(cfg)
summary = model.summary()
print(f"✓ Model constructed: {summary['n_units']} neurons")

## Step 2: Setup Simulation

In [ ]:
sim = jtfne.Simulation(
    duration_ms=DURATION_MS,
    dt_ms=DT_MS,
    seed=SEED,
    record_sources=True,
    record_fields=True,
)
print(f"✓ Simulation object created")

## Step 3: Baseline Simulation

In [ ]:
print(f"Running baseline simulation...")
signals_baseline = model.simulate(sim)
baseline_summary = signals_baseline.summary()
baseline_rate = baseline_summary["spike_rate_hz_mean"]
baseline_kappa = jtfne.kappa_synchrony(signals_baseline.spikes, dt_ms=DT_MS)
print(f"✓ Baseline: rate={baseline_rate:.2f}Hz, kappa={baseline_kappa:.3f}")

## Step 4: Targeted Stimulus

In [ ]:
target_indices = jtfne.select_neurons(model, area="V1", cell_type="E")
if len(target_indices) == 0:
    target_indices = jtfne.select_neurons(model, area="V1")

stim = jtfne.stimulus_schedule(
    [{"label": "stim", "onset_ms": 100, "duration_ms": 100, 
      "amplitude": 1.5, "target_indices": target_indices.tolist()}],
    n_neurons=summary["n_units"]
)
print(f"✓ Stimulus created for {len(target_indices)} neurons")

signals_stim = model.simulate(sim, paradigm=stim)
stim_rate = signals_stim.summary()["spike_rate_hz_mean"]
stim_kappa = jtfne.kappa_synchrony(signals_stim.spikes, dt_ms=DT_MS)
print(f"✓ Stimulus: rate={stim_rate:.2f}Hz, kappa={stim_kappa:.3f}")

## Step 5: AGSDR Tuning

In [ ]:
objective = jtfne.rate_synchrony_targets(
    target_rate_hz=3.5,
    target_kappa_synchrony=0.0,
    rate_weight=1.0,
    synchrony_weight=0.25,
)

optimizer = jtfne.agsdr(
    parameters={"noise_amplitude": (0.1, 1.0)},
    generations=3,
    population_size=2,
    seed=SEED,
)

print(f"Starting AGSDR tuning...")
tuned = model.tune(
    objectives=objective,
    optimizer=optimizer,
    simulation=sim,
    seed=SEED,
)
print(f"✓ Tuning complete")

## Step 6: Analysis

In [ ]:
tuned_model = tuned.model
signals_tuned = tuned_model.simulate(sim, paradigm=stim)
tuned_rate = signals_tuned.summary()["spike_rate_hz_mean"]
tuned_kappa = jtfne.kappa_synchrony(signals_tuned.spikes, dt_ms=DT_MS)

df = pd.DataFrame([
    {"Condition": "Baseline", "Rate (Hz)": baseline_rate, "Kappa": baseline_kappa},
    {"Condition": "Stimulus", "Rate (Hz)": stim_rate, "Kappa": stim_kappa},
    {"Condition": "Tuned+Stim", "Rate (Hz)": tuned_rate, "Kappa": tuned_kappa},
])
print(df.to_string(index=False))

## Step 7: Visualization

In [ ]:
fig = jtfne.vis.spectrolaminar_suite(
    signals_tuned,
    freq_min_hz=1.0,
    freq_max_hz=150.0,
    psd_nperseg=128,
    figsize=(12, 8),
    dpi=150,
    title="Etude No. 1: Spectrolaminar Profile"
)
fig_path = FIG_DIR / "spectrolaminar.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"✓ Saved: {fig_path}")

## Step 8: Export Artifacts

In [ ]:
manifest = {
    "run_id": RUN_ID,
    "seed": SEED,
    "duration_ms": DURATION_MS,
    "baseline_rate_hz": float(baseline_rate),
    "tuned_rate_hz": float(tuned_rate),
    "target_rate_hz": 3.5,
}

jtfne.save_json(manifest, OUTPUT_DIR / "manifest.json")
validation = {
    "tutorial_id": RUN_ID,
    "truth_mode": "truth_safe_unverified",
    "claim_level": "computational_scaffold",
}
jtfne.save_json(validation, OUTPUT_DIR / "validation_report.json")

print(f"✓ Artifacts exported to {OUTPUT_DIR}")
print(f"✅ ETUDE NO. 1 COMPLETE")